## Comparing the PCA workflow to Burns' weighted log-odds method

This notebook compares our existing PCA + logistic-regression "speechiness" score (`ccc2026.run_training`/`rolling_samples`) against Burns' weighted log-odds "dialogism" method (`ccc2026.dialogism`), using each method's finished, packaged implementation. See `3 - Burns' dialogism method` for how the dialogism method itself was built and verified.

In [1]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

import ccc2026
from ccc2026 import dialogism
ccc2026.setup()

tokens = ccc2026.tokens

### Compute the dialogism score

Using the packaged, already-verified implementation (see `3 - Burns' dialogism method`).

In [2]:
lexicons = dialogism.build_lexicons()
dialogism_score = dialogism.token_dialogism_score(lexicons)
dialogism_score.describe()

count    429774.000000
mean          0.392374
std           0.120016
min           0.086293
25%           0.322335
50%           0.361753
75%           0.417210
max           0.898011
dtype: float64

### Rolling dialogism score vs. the PCA + logistic regression score

Burns uses a rolling window of 50 tokens for his narrative plot (Figure 9.4); our existing `rolling_samples` defaults to 500. To compare fairly, both methods need to use the *same* window size — so the cell below recomputes the PCA score at each window size tested, rather than reusing a single fixed-window PCA score.

In [3]:
# existing PCA + logistic regression speechiness score, for comparison
feature_set = {
    "lemma": ccc2026.top_lemmas,
    "pos": ccc2026.all_pos,
    "morph": ccc2026.top_morph,
}
train = ccc2026.run_training(feature_set)

In [4]:
for window in [50, 200, 500, 1000]:
    dialogism_roll = dialogism.rolling_dialogism(dialogism_score, window_size=window)["speech_score"]["score"]

    # PCA score recomputed at the same window, so the comparison is apples-to-apples
    pca_score = ccc2026.rolling_samples(train, window_size=window)["speech_score"]["score"]

    both = pd.DataFrame({"dialogism": dialogism_roll, "pca": pca_score}).dropna()
    print(
        f"window={window}: n={len(both)}, "
        f"pearson r={both['dialogism'].corr(both['pca']):.3f}, "
        f"spearman r={both['dialogism'].corr(both['pca'], method='spearman'):.3f}"
    )

window=50: n=427589, pearson r=0.876, spearman r=0.868
window=200: n=420689, pearson r=0.897, spearman r=0.896
window=500: n=406889, pearson r=0.891, spearman r=0.887
window=1000: n=383889, pearson r=0.876, spearman r=0.872


### Pick one window size for the rest of this notebook

Agreement is strong and stable across window sizes (Pearson r &asymp; 0.87&ndash;0.89 from 50 to 1000 tokens) &mdash; confirming the intuition that the two methods are largely measuring the same thing. The three views below all use one shared window, computed once here, rather than each view picking its own. Change `WINDOW` and re-run from here to see all three update together.

In [ ]:
WINDOW = 200

dialogism_roll = dialogism.rolling_dialogism(dialogism_score, window_size=WINDOW)["speech_score"]["score"]
pca_roll = ccc2026.rolling_samples(train, window_size=WINDOW)["speech_score"]["score"]

### View 1: paired scores, per window

Each point is one rolling window; x and y are the two methods' raw scores for that same window. A tight, roughly linear cloud is what "the two methods agree" looks like at the level of individual windows, not just in aggregate correlation.

In [ ]:
both = pd.DataFrame({"dialogism": dialogism_roll, "pca": pca_roll}).dropna()

g = sns.relplot(data=both, x="pca", y="dialogism", alpha=0.1, height=4)
g.set(
    xlabel="PCA + logistic regression score",
    ylabel="weighted log-odds dialogism score",
    title=f"window={WINDOW}",
)
plt.show()

### View 2: distribution by narrative class

Same windows, same two scores — but now split by whether the window falls in speech, narration, or Odysseus' apologue (kept separate as "other", same as elsewhere in the package). Both methods should score "speech" systematically higher than "narration"; comparing the two colors within each category shows whether they agree not just on average, but on the shape of the distribution.

In [ ]:
# narrative-class label per token, independent of window or method
label = pd.Series("speech", index=tokens.index)
label[tokens["speaker"] == "Odysseus-Apologue"] = "other"
label[tokens["speaker"].isna()] = "narration"

# z-scored so both methods are on the same axis
z_scores = pd.DataFrame({
    "label": label,
    "PCA + logistic regression": (pca_roll - pca_roll.mean()) / pca_roll.std(),
    "weighted log-odds dialogism": (dialogism_roll - dialogism_roll.mean()) / dialogism_roll.std(),
}).melt(id_vars="label", var_name="method", value_name="z_score").dropna()

# a swarm plot isn't practical at ~430k points, so use a split violin instead —
# same "two colors side by side per category" idea, as a density shape rather
# than individual dots
g = sns.catplot(
    data=z_scores,
    x="label", y="z_score", hue="method",
    kind="violin", split=True, inner="quartile",
    order=["narration", "speech", "other"],
    aspect=2, height=5,
)
g.set(ylabel="standardized score", xlabel="")
g._legend.set_title("")
plt.show()

### View 3: mapped onto the poem itself

The first two views treat windows as an unordered collection of points. This one restores the thing a reader actually cares about — sequence: both scores, standardized, plotted line by line across a single book, so you can see where the two methods rise and fall together as the narrative unfolds.

In [ ]:
work, pref = "Dionysiaca", "1"
mask = (tokens["work"] == work) & (tokens["pref"] == pref)
idx = tokens.index[mask]

book = pd.DataFrame({
    "dialogism": dialogism_roll.reindex(idx),
    "pca": pca_roll.reindex(idx),
    "line": tokens.loc[idx, "line"],
}).dropna()

# standardize both scores so they're comparable on one axis
book["dialogism_z"] = (book["dialogism"] - book["dialogism"].mean()) / book["dialogism"].std()
book["pca_z"] = (book["pca"] - book["pca"].mean()) / book["pca"].std()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(book.index, book["pca_z"], label="PCA + logistic regression (standardized)")
ax.plot(book.index, book["dialogism_z"], label="weighted log-odds dialogism (standardized)")
ax.axhline(0, color="k", ls="--", lw=1)
ax.set_title(f"{work} {pref}")
ax.set_xlabel("token index")
ax.legend()
plt.show()